# Day 4 | Lab 4.4: Amazon Bedrock Knowledge Bases — Managed RAG

**Duration:** ~1 hour

**Scenario.** Same retail-banking policy corpus from Lab 4.3 — but now AWS handles the entire RAG pipeline (chunking, embedding, vector store, retrieval) for you via **Bedrock Knowledge Bases**. You point it at an S3 bucket of policy PDFs and call `retrieve` or `retrieve_and_generate`.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Decide between **managed RAG** (Bedrock KB) and **self-hosted RAG** (Lab 4.3) given a real production scenario.
2. Use the Bedrock `Retrieve` API to get raw chunks from a KB — same role as a LangChain retriever.
3. Use the Bedrock `RetrieveAndGenerate` API for end-to-end RAG with a Bedrock-hosted Claude model.
4. Combine a Bedrock KB with a Bedrock **Guardrail** (preview of Lab 6.3 — full coverage there).
5. Apply a production decision matrix — when to build vs buy, what data residency / cost / vendor-lock factors to weigh.

**Tools.** AWS Bedrock (`bedrock-agent-runtime` for KB, `bedrock-runtime` for direct invokes) · `boto3`. Pre-requisite: a Bedrock Knowledge Base already created in the AWS Console (one-time setup, ~10 minutes).

*Extracted from GM Module 10 / AWS Labs / `Module10_Lab7_Bedrock_KnowledgeBases_Guardrails.ipynb` — KB sections (8–11) only. Full Guardrails coverage is in Lab 6.3 (Module 6). Created by Prashant Sahu · [LinkedIn](https://www.linkedin.com/in/prashantksahu/)*

---


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q boto3 pandas


## 1. AWS Credentials

In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 2. Initialise Bedrock Clients

Two distinct Bedrock clients are involved:
- **`bedrock-runtime`** — direct model invocation (`invoke_model`, `converse`).
- **`bedrock-agent-runtime`** — Knowledge Base APIs (`retrieve`, `retrieve_and_generate`) and Bedrock Agents.


In [3]:
import boto3
import json
import time

# Bedrock clients
bedrock_client = boto3.client('bedrock')           # For creating guardrails
bedrock_runtime = boto3.client('bedrock-runtime')   # For invoke/converse/apply_guardrail
bedrock_agent = boto3.client('bedrock-agent')       # For KB management
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')  # For KB queries

print("✅ Bedrock client (management)")
print("✅ Bedrock Runtime (inference + guardrails)")
print("✅ Bedrock Agent (KB management)")
print("✅ Bedrock Agent Runtime (KB queries)")

✅ Bedrock client (management)
✅ Bedrock Runtime (inference + guardrails)
✅ Bedrock Agent (KB management)
✅ Bedrock Agent Runtime (KB queries)


---
## 3. NEW: Managed RAG (Bedrock KB) vs Self-Hosted RAG (Lab 4.3)

Lab 4.3 you built it from scratch — chunking, embedding, vector store, retrieval, prompting — all in your code. Bedrock Knowledge Bases hands all of that off to AWS in exchange for vendor lock-in. When does each win?

| Dimension | Self-Hosted RAG (Lab 4.3) | Bedrock Knowledge Base (this lab) |
|---|---|---|
| Setup effort | High — write loaders, splitters, retriever, chain | Low — point at S3, click "Sync", get an ARN |
| Operational burden | You run the vector store, embedding pipeline, retrieval | AWS runs all of it |
| Customisation | Total control over every step | Limited — chunking strategy + a few overrides |
| Embedding model choice | Any (OpenAI, Titan, BGE, custom-fine-tuned) | Bedrock-supported only (Titan, Cohere) |
| Vector store | Any (FAISS, Chroma, pgvector, OpenSearch, Pinecone, etc.) | OpenSearch Serverless / Aurora / Pinecone / Redis Enterprise (managed) |
| Chunking control | Full — section-aware, custom splitters | Fixed strategies (fixed-size, hierarchical, semantic) + chunk-size knob |
| Retrieval algorithm | Any (MMR, contextual, hybrid, re-ranking) | Hybrid (semantic + keyword) plus optional re-ranker |
| Cost model | Per-query embedding + your vector-store hosting | Per-query + OpenSearch Serverless OCU/storage hourly |
| Latency | Sub-second (in-memory FAISS) to ~200 ms | ~300–800 ms typical |
| Data residency | Wherever you deploy | Bedrock region only |
| Vendor lock-in | None | High — Bedrock-specific APIs |
| Best for | High-customisation requirements, multi-cloud, edge | AWS-only stack, fast time-to-prototype, ops-light teams |

> **Default rule.**
> - Stack is on AWS, you want a working RAG by Friday → **Bedrock KB**.
> - You need custom chunking / non-standard embeddings / re-ranking / multi-cloud / on-prem → **Self-hosted (Lab 4.3 + Module 5).**


---
## 4. Knowledge Base ID Configuration

**Pre-requisite (one-time setup, in the AWS Console).** Create a Bedrock Knowledge Base:
1. Bedrock Console → Knowledge Bases → Create.
2. Data source: an S3 bucket containing your PDFs / Markdown / Word files.
3. Embedding model: `Titan Embeddings G1` or `Titan Text Embeddings v2`.
4. Vector store: AWS picks OpenSearch Serverless by default (or Aurora / Pinecone / Redis Enterprise).
5. Sync — wait ~5–10 minutes for ingestion.
6. Copy the **KB ID** (e.g., `1A2B3C4D5E`) and paste into the next cell.

Detailed walkthrough: [AWS Bedrock KB docs](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base-create.html).

In [8]:
# 🔧 Set your Knowledge Base ID here (from the AWS Console)
KNOWLEDGE_BASE_ID = "AFT2FSLYOP"  # e.g., "ABCDEFGHIJ"

# Model ARN for retrieve_and_generate
MODEL_ARN = f"arn:aws:bedrock:{AWS_REGION}::foundation-model/{BEDROCK_MODEL}"

if KNOWLEDGE_BASE_ID:
    print(f"✅ Knowledge Base ID: {KNOWLEDGE_BASE_ID}")
    print(f"   Model ARN: {MODEL_ARN}")
else:
    print("⚠️ No Knowledge Base ID set — running in demo/simulation mode")
    print("   Set KNOWLEDGE_BASE_ID above to use a real Knowledge Base")

✅ Knowledge Base ID: AFT2FSLYOP
   Model ARN: arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-lite-v1:0


---
## 5. The `Retrieve` API — Raw Chunks from the KB

`retrieve` takes a query, returns the top-N chunks with text + source metadata + similarity scores. Same role as `vectorstore.as_retriever()` in Lab 4.3 — but AWS does the embedding + search for you. Use this when you want to plug the KB into your own custom prompt / chain / re-ranker.

In [16]:
def retrieve_from_kb(query, kb_id, max_results=5):
    """
    Retrieve relevant documents from a Bedrock Knowledge Base.
    Returns the retrieved chunks with source info and confidence.
    """
    if not kb_id:
        print("⚠️ No Knowledge Base ID configured — showing demo output")
        # Simulated response for learning purposes
        demo_results = [
            {"content": "SafeGuard Health Premier policy covers cardiac procedures including PTCA, CABG, and valve replacement up to the sum insured limit. Pre-authorization is required for procedures exceeding Rs. 1,00,000.",
             "source": "s3://safeguard-docs/policy_guidelines_health.pdf",
             "score": 0.92},
            {"content": "Emergency hospitalization claims must be intimated within 24 hours of admission. For planned procedures, pre-authorization must be obtained at least 48 hours before admission.",
             "source": "s3://safeguard-docs/claims_process.pdf",
             "score": 0.87},
        ]
        print(f"\n📚 Retrieved {len(demo_results)} chunks (DEMO):")
        for i, r in enumerate(demo_results):
            print(f"\n  [{i+1}] Score: {r['score']:.2f}")
            print(f"      Source: {r['source']}")
            print(f"      Content: {r['content'][:150]}...")
        return demo_results

    response = bedrock_agent_runtime.retrieve(
        knowledgeBaseId=kb_id,
        retrievalQuery={'text': query},
        retrievalConfiguration={
            'vectorSearchConfiguration': {
                'numberOfResults': max_results,
            }
        },
    )

    results = []
    for chunk in response.get('retrievalResults', []):
        content = chunk.get('content', {}).get('text', '')
        source = chunk.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
        score = chunk.get('score', 0)
        results.append({'content': content, 'source': source, 'score': score})
        print(f"\n  Score: {score:.2f} | Source: {source}")
        print(f"  Content: {content[:150]}...")

    print(f"\n📚 Retrieved {len(results)} chunks")
    return results

# Query the knowledge base
results = retrieve_from_kb(
    "What is the difference between generative AI and traditional AI?",
    KNOWLEDGE_BASE_ID,
)


  Score: 0.74 | Source: s3://prashant-rag-1/Introduction_to_GENAI (1).html
  Content: ratization
User-friendly interfaces made AI accessible to everyone
# Traditional AI vs Generative AI
Aspect Traditional AI Generative AI
| Aspect | Tr...

  Score: 0.64 | Source: s3://prashant-rag-1/Introduction_to_GENAI (1).html
  Content: than just analyzing existing data # What is Generative AI? Generative AI is a subset of artificial intelligence that can create new content, including...

  Score: 0.64 | Source: s3://prashant-rag-1/Introduction_to_GENAI (1).html
  Content: patterns from massive datasets * **Powered by DL:** Leverages advanced neural network architectures (like Transformers) * **Includes NLP:** Many GenAI...

  Score: 0.63 | Source: s3://prashant-rag-1/Introduction_to_GENAI (1).html
  Content: ? Generative AI is a subset of artificial intelligence that can create new content, including text, images, audio, video, and code, based on the patte...

  Score: 0.58 | Source: s3://prasha

---
## 6. The `RetrieveAndGenerate` API — End-to-End Managed RAG

`retrieve_and_generate` is the one-shot RAG endpoint: query in → final answer + citations out. AWS handles retrieval AND prompt construction AND LLM call. Pick a Claude model on Bedrock for the generation step.

**Pros.** Fastest path to a working RAG — single API call, citations included.
**Cons.** No control over the prompt; tightly coupled to Bedrock-hosted models; harder to debug retrieval quality.

In [17]:
def rag_query(query, kb_id, model_arn=None, session_id=None):
    """
    Query a Knowledge Base with retrieve_and_generate (full RAG).
    Returns generated response with citations.
    """
    if model_arn is None:
        model_arn = MODEL_ARN

    if not kb_id:
        print("⚠️ No Knowledge Base ID — showing demo RAG response")
        demo = {
            "response": (
                "Based on the SafeGuard Health Premier policy guidelines, cardiac stent "
                "procedures (PTCA with DES placement) are covered up to the full sum insured "
                "limit. Pre-authorization is required for procedures exceeding Rs. 1,00,000. "
                "Emergency cases are exempt from pre-authorization requirements, but must be "
                "intimated within 24 hours of admission. [Source: policy_guidelines_health.pdf]"
            ),
            "citations": [
                {"source": "s3://safeguard-docs/policy_guidelines_health.pdf",
                 "text": "Cardiac procedures including PTCA..."},
                {"source": "s3://safeguard-docs/claims_process.pdf",
                 "text": "Emergency hospitalization claims..."},
            ]
        }
        print(f"\n🤖 RAG Response (DEMO):\n{demo['response']}")
        print(f"\n📎 Citations: {len(demo['citations'])}")
        for c in demo['citations']:
            print(f"   → {c['source']}")
        return demo

    params = {
        'input': {'text': query},
        'retrieveAndGenerateConfiguration': {
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': model_arn,
            },
        },
    }
    if session_id:
        params['sessionId'] = session_id

    response = bedrock_agent_runtime.retrieve_and_generate(**params)

    # Extract response text
    output_text = response.get('output', {}).get('text', '')
    session = response.get('sessionId', '')

    # Extract citations
    citations = []
    for citation in response.get('citations', []):
        refs = citation.get('retrievedReferences', [])
        for ref in refs:
            source = ref.get('location', {}).get('s3Location', {}).get('uri', 'unknown')
            content = ref.get('content', {}).get('text', '')
            citations.append({'source': source, 'text': content[:100]})

    print(f"\n🤖 RAG Response:\n{output_text}")
    print(f"\n📎 Citations: {len(citations)}")
    for c in citations:
        print(f"   → {c['source']}")

    return {'response': output_text, 'citations': citations, 'session_id': session}

# Test RAG queries
print("=" * 60)
print("📚 Knowledge Base RAG Queries")
print("=" * 60)

queries = [
    "What documents are needed to file a health insurance claim?",
    "Is pre-authorization required for emergency cardiac procedures?",
    "What is the claims settlement timeline for motor insurance?",
]

for q in queries:
    print(f"\n❓ Query: {q}")
    rag_query(q, KNOWLEDGE_BASE_ID)
    print()

📚 Knowledge Base RAG Queries

❓ Query: What documents are needed to file a health insurance claim?

🤖 RAG Response:
Answer: To file a health insurance claim, you need the following documents:

1. Duly filled claim form (signed by insured and treating doctor) 2. Original hospital bills and receipts (itemized) 3. Discharge summary from the hospital 4. Investigation reports (lab tests, imaging) 5. Prescription copies 6. KYC documents (Aadhaar or PAN copy) 7. Cancelled cheque or bank account details 8. FIR/MLC report (for accident cases) These documents are required for reimbursement claims. For cashless claims, you need the health card/policy number, doctor's referral/recommendation, and pre-authorization form.

📎 Citations: 9
   → s3://prashant-rag-1/insurance_policy_documents/04_claims_faq.md
   → s3://prashant-rag-1/insurance_policy_documents/04_claims_faq.md
   → s3://prashant-rag-1/insurance_policy_documents/04_claims_faq.md
   → s3://prashant-rag-1/insurance_policy_documents/04_clai

### NEW: Worked Example with Claude Sonnet 4.5 on Bedrock

Below is a complete `retrieve_and_generate` invocation with explicit Claude model ARN, showing the full response shape — `output.text` (the answer), `citations` (source chunks), and `sessionId` (for follow-up turns within the same KB session).

In [ ]:
import json

# Use a Bedrock cross-region inference profile for higher availability.
# Verify current model IDs at: https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html
CLAUDE_MODEL_ARN = (
    f'arn:aws:bedrock:{os.environ.get("AWS_DEFAULT_REGION", "us-east-1")}::'
    'foundation-model/anthropic.claude-sonnet-4-5-20250929-v1:0'
)

def rag_query_with_claude(query: str, kb_id: str, session_id: str | None = None):
    """End-to-end managed RAG: retrieve from KB + generate answer with Claude."""
    request = {
        'input': {'text': query},
        'retrieveAndGenerateConfiguration': {
            'type': 'KNOWLEDGE_BASE',
            'knowledgeBaseConfiguration': {
                'knowledgeBaseId': kb_id,
                'modelArn': CLAUDE_MODEL_ARN,
                'retrievalConfiguration': {
                    'vectorSearchConfiguration': {'numberOfResults': 4},
                },
            },
        },
    }
    if session_id:
        request['sessionId'] = session_id
    return bedrock_agent_runtime.retrieve_and_generate(**request)

# Run it (replace KB_ID with your real KB id from Section 4).
# response = rag_query_with_claude('What is the prepayment fee on a personal loan?', kb_id=KB_ID)
# print('Answer:')
# print(response['output']['text'])
# print()
# print('Citations:')
# for cit in response.get('citations', []):
#     for ref in cit.get('retrievedReferences', []):
#         print(' -', ref['location'].get('s3Location', {}).get('uri'))
#         print('   chunk:', ref['content']['text'][:150], '...')
# print()
# print('Session ID (for follow-up turns):', response.get('sessionId'))


---
## 7. Knowledge Base + Guardrails (Preview)

Bedrock Guardrails (denied topics, content filters, PII redaction) plug into `retrieve_and_generate` via `guardrailConfiguration`. The guardrail evaluates both the retrieved context and the model's draft output, and intervenes if either crosses your policy.

**Full coverage of Guardrails — including how to author them, the 6 content-filter categories, and the `apply_guardrail` API for standalone use — is in Lab 6.3 (Module 6: Production Patterns).** Below is a preview showing how to wire an existing Guardrail to your KB-RAG call.

In [18]:
def safe_rag_query(query, kb_id, guardrail_id, guardrail_version):
    """
    RAG query with guardrail protection on both input and output.
    """
    print(f"❓ Query: {query}")

    # Step 1: Screen the input with guardrail
    print("  Step 1: Screening input...")
    input_check = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source='INPUT',
        content=[{'text': {'text': query}}],
    )

    if input_check['action'] == 'GUARDRAIL_INTERVENED':
        blocked_msg = input_check['outputs'][0]['text'] if input_check.get('outputs') else "Query blocked"
        print(f"  🛑 Input blocked: {blocked_msg}")
        return {'response': blocked_msg, 'blocked': True}

    print("  ✅ Input passed guardrail")

    # Step 2: RAG query
    print("  Step 2: Querying Knowledge Base...")
    rag_result = rag_query(query, kb_id)

    # Step 3: Screen the output with guardrail
    print("  Step 3: Screening output...")
    output_check = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion=guardrail_version,
        source='OUTPUT',
        content=[{'text': {'text': rag_result['response']}}],
    )

    if output_check['action'] == 'GUARDRAIL_INTERVENED':
        # Use the guardrail's sanitized output instead
        safe_output = output_check['outputs'][0]['text'] if output_check.get('outputs') else rag_result['response']
        print(f"  🛡️ Output modified by guardrail")
        return {'response': safe_output, 'blocked': False, 'modified': True}

    print("  ✅ Output passed guardrail")
    return {'response': rag_result['response'], 'blocked': False, 'modified': False}

# Test safe RAG with both allowed and blocked queries
print("=" * 60)
print("🛡️ + 📚  Safe RAG (Knowledge Base + Guardrail)")
print("=" * 60)

safe_queries = [
    "What is covered under the health premier policy?",           # Safe
    "Compare SafeGuard with HDFC Ergo health plans",              # Blocked (competitor)
    "Should I invest my claim payout in mutual funds?",           # Blocked (investment)
    "What is the claims process for motor accidents?",            # Safe
]

for q in safe_queries:
    print(f"\n{'─' * 40}")
    result = safe_rag_query(q, KNOWLEDGE_BASE_ID, GUARDRAIL_ID, GUARDRAIL_VERSION)

🛡️ + 📚  Safe RAG (Knowledge Base + Guardrail)

────────────────────────────────────────
❓ Query: What is covered under the health premier policy?
  Step 1: Screening input...
  ✅ Input passed guardrail
  Step 2: Querying Knowledge Base...

🤖 RAG Response:
Answer: The Health Premier policy covers the following :

- **In-Patient Hospitalization:** Treatment requiring admission to a hospital for a minimum of 24 hours. This includes room rent and boarding charges, nursing charges, surgeon, anesthetist, and consultant fees, anesthesia, blood, oxygen, operation theatre charges, medicines and drugs, diagnostic tests during hospitalization, and ICU/CCU charges (no sub-limit in Premier and Gold plans).
- **Pre-Hospitalization Expenses:** Medical expenses incurred up to 60 days before the date of admission, directly related to the diagnosis for which hospitalization was required.
- **Restoration Benefit:** 200% restoration, unlimited.
- **International Emergency Cover:** Up to $25,000 (emergency

---
## 8. NEW: Production Decision Matrix — Build vs Buy

When you sit in front of a client and they ask *"do we use Bedrock KB or build it ourselves?"* — these are the seven dimensions worth weighing:

| Question | Lean Bedrock KB | Lean Self-Hosted (Lab 4.3 + Module 5) |
|---|---|---|
| **1. Where does the data live?** | Already on S3, IAM-controlled | On-prem, GCP, multi-cloud, or non-AWS regions |
| **2. How custom is your retrieval?** | Standard semantic search is fine | Need MMR / contextual / hybrid / custom re-ranker |
| **3. How fast does the corpus change?** | Sync nightly is fine | Updates per minute; real-time ingestion |
| **4. What's the team's ops capacity?** | Small / no ML-ops team | Have ML-ops capacity to run a vector store |
| **5. What's the answer-quality bar?** | "Good enough," Bedrock defaults work | Need to fine-tune chunking, embeddings, re-ranking against an eval set |
| **6. What's the total spend (rough)?** | < 100K queries/month: KB pricing wins | > 1M queries/month: self-hosted often cheaper at scale |
| **7. What about compliance / data residency?** | Bedrock region matches your jurisdiction | Need on-prem or non-Bedrock regions |

**Common path.** Prototype in Bedrock KB to validate the use case (week 1). If retrieval quality holds, stay there. If you hit ceilings on customisation or cost at scale, migrate to a self-hosted stack — the code from Lab 4.3 + Module 5 ports directly.

**Red flags that say move OFF Bedrock KB:**
- Recall@5 < 70% on your eval set after tuning chunk size in the console — you've hit the ceiling of fixed strategies.
- Need to re-rank with a custom cross-encoder — KB doesn't expose an API hook.
- Per-query latency > 1 sec consistently — you may need a faster vector store closer to your service.
- Multi-cloud or hybrid — Bedrock KB is region/account-locked.


---
## 9. Conclusion & Key Takeaways

### What We Covered

| Concept | Takeaway |
|---|---|
| **Bedrock KB = managed end-to-end RAG** | S3 + Console + a KB ID = working RAG; AWS owns the pipeline |
| **`retrieve` vs `retrieve_and_generate`** | First gives raw chunks for your custom chain; second gives the full RAG answer + citations |
| **Cross-region inference profiles for Claude** | Use cross-region ARNs for higher availability and broader capacity |
| **Citations come back automatically** | `response['citations']` carries the S3 source + chunk text — no extra wiring |
| **Guardrails plug in via `guardrailConfiguration`** | Full coverage in Lab 6.3 — content filters, denied topics, PII |
| **Decision is mostly about ops cost vs customisation** | Bedrock for fast time-to-value on AWS; self-hosted for full control or non-AWS |

**Next Lab:** Module 5 — Advanced RAG & Evaluation 🎯


## 10. Stretch Exercise (Optional)

1. Set up a Bedrock KB in your AWS sandbox account (S3 → Bedrock Console → 5 PDFs → Sync). Compare retrieval results against your Lab 4.3 self-hosted FAISS index over the same 10-question eval set.
2. Build a multi-turn conversation using `sessionId` — verify the second query references context from the first turn.
3. Try all three Bedrock chunking strategies on the same source: fixed-size, hierarchical, semantic. Compare retrieval quality on the same eval set.
4. Combine the KB with a Bedrock Guardrail (denied topic: "investment advice"). Verify a query like "should I buy NIFTY futures?" gets blocked.
5. Build a *fallback path*: try `retrieve_and_generate` first; if `numberOfRetrievedResults == 0`, fall back to a plain Claude call with a polite "I don't have that information" prompt.
6. Pull the Bedrock KB pricing page and compute the breakeven query volume vs your self-hosted equivalent (OpenSearch Serverless OCUs + embedding API). When does Bedrock win?


---

## Interview Preparation

The questions below mirror what client interviewers commonly ask about the topics in this lab. Use the hint to think through the answer first; use the sketch only to verify your reasoning.

---

**Q1. When should you choose Bedrock Knowledge Bases over a self-hosted RAG?**

*Hint:* Time-to-value vs customisation.

*Answer sketch:* Choose Bedrock KB when: data is on AWS S3, you want a working RAG by end of week, your team is small/ops-light, standard semantic search is good enough, and you accept Bedrock vendor lock-in. Choose self-hosted (Lab 4.3 + Module 5) when: you need custom chunking / re-ranking / non-Bedrock embeddings, multi-cloud or on-prem deployment, sub-200ms latency, or query volume so high that managed pricing dominates total cost.

---

**Q2. What does Bedrock KB do under the hood — what's the actual pipeline?**

*Hint:* Same RAG pipeline you build by hand, but managed.

*Answer sketch:* On ingest: AWS pulls files from your S3 source → splits with the configured chunking strategy → embeds each chunk with the configured Titan/Cohere model → upserts into the configured vector store (OpenSearch Serverless by default; Aurora / Pinecone / Redis Enterprise as alternatives). On query: embed the query → vector search (hybrid: semantic + keyword) → optional re-ranker → return top-N. `retrieve_and_generate` adds: stuff into a prompt → call the chosen Bedrock model → return answer + citations.

---

**Q3. How does Bedrock KB handle chunking — can you control it?**

*Hint:* Three strategies, plus a chunk-size knob.

*Answer sketch:* Three built-in strategies: (1) **Fixed-size** — N tokens with overlap, simplest. (2) **Hierarchical** — parent/child chunks; small for retrieval, large for context. (3) **Semantic** — embedding-aware boundary detection. You set the strategy + chunk size + overlap in the console at KB creation time. You can NOT plug in a custom splitter — that's the customisation ceiling. If you need section-aware Markdown splitting (Lab 4.1 §5), you have to self-host.

---

**Q4. What's the cost model — per-query vs per-document?**

*Hint:* Three line items.

*Answer sketch:* Three components: (1) **Embedding cost** — per-token, charged on every ingest and every query. (2) **Vector-store hosting** — for OpenSearch Serverless: hourly OCU + storage GB-month (this is the surprise line item — minimum 2 OCUs ≈ $700/mo even if idle). (3) **Generation cost** — per-token charged on `retrieve_and_generate` (Claude / Llama / Titan etc.). For low-volume KBs with tiny corpora, OCU minimum dominates total cost; at scale, generation tokens dominate.

---

**Q5. How do you keep a KB in sync with a constantly-changing source S3 bucket?**

*Hint:* Manual sync, scheduled sync, or event-driven sync.

*Answer sketch:* Three options: (1) **Manual** — click "Sync" in the console (fine for static or rarely-updated KBs). (2) **Scheduled** — `start_ingestion_job` from a CloudWatch Events / EventBridge cron (typical: nightly). (3) **Event-driven** — S3 → EventBridge → Lambda → `start_ingestion_job` on every object PUT (typical for fast-moving content). Note: Bedrock re-ingests changed/added/deleted files on each sync — incremental, but charged per-token on changed content only.

---

**Q6. What are the trade-offs between `retrieve` and `retrieve_and_generate` APIs?**

*Hint:* Raw chunks vs end-to-end answer.

*Answer sketch:* `retrieve` returns raw chunks + scores + citations — use when you want to build your own prompt, plug into a LangChain chain, do custom re-ranking, or use a non-Bedrock LLM. `retrieve_and_generate` is the one-shot RAG endpoint — easiest path, returns answer + citations, but tightly coupled to Bedrock LLMs and uses AWS's prompt template (not customisable). Default: prototype with `retrieve_and_generate`, switch to `retrieve` + your own chain when you need custom prompting or non-Bedrock generation.

---

**Q7. How do you debug poor retrieval quality in a managed KB you don't fully control?**

*Hint:* Log everything, then twiddle the knobs you have.

*Answer sketch:* Step 1: log every `retrieve` response (chunks + scores) and the user query — find the failing queries. Step 2: try the same query with `numberOfResults=10` to see if the answer is just deeper in the list (if yes, raise k or add a re-ranker). Step 3: check whether the source doc was even ingested (KB Console → Data Source → file status). Step 4: try a different chunking strategy (fixed → hierarchical → semantic) — re-sync, re-test. Step 5: still failing → switch to self-hosted (Lab 4.3 + Module 5 advanced retrieval).

